# Evaluate one model on one problem

Pull one row from the dataset, ask a model through OpenRouter, and grade the reply with the
family's own `parse_answer()` and `verify()` from this repo. `verify()` is the grader, not
string equality against the gold `answer` column: many families accept more than one valid
answer, and `verify()` is the same function that graded the dataset when it was built.

Needs `OPENROUTER_API_KEY` in the environment and `pip install huggingface_hub`. Run it from
where it lives, `notebooks/` inside the repo.

## 1. Configuration

In [1]:
import os, sys, json, csv, random, glob, importlib.util, io, contextlib, urllib.request

HF_DATASET = "amphora/math-intuition-20260905-408-easy-30"
CSV_NAME   = "math-intuition-20260905-408-easy-30.csv"
REPO_DIR   = os.path.abspath("..")    # this notebook lives in <repo>/notebooks/

MODEL      = "openai/gpt-5.6-terra"   # any OpenRouter model id
EFFORT     = "medium"                 # reasoning effort
MAX_TOKENS = 32000

ROW_INDEX  = None                     # None -> random row, or an integer to pin one
PICK_PAPER = None                     # None, or an arXiv id to target one family
SEED       = 0                        # controls which row is picked, not the problem

csv.field_size_limit(sys.maxsize)     # some questions exceed the 128 KB default

131072

## 2. Pick one row

In [2]:
from huggingface_hub import hf_hub_download

rows = list(csv.DictReader(open(hf_hub_download(HF_DATASET, CSV_NAME, repo_type="dataset"), encoding="utf-8")))
pool = [r for r in rows if r["paper"] == PICK_PAPER] if PICK_PAPER else rows
row  = pool[ROW_INDEX] if ROW_INDEX is not None else random.Random(SEED).choice(pool)

print(f"{row['id']}   arXiv:{row['paper']}  preset={row['preset']}  seed={row['seed']}\n")
print(row["question"][:1500])

2103.10687::11   arXiv:2103.10687  preset=easy  seed=1537388980

Find a normalized differential witness for a piecewise Dobbertin permutation.

Binary polynomials are encoded as hexadecimal coefficient masks: bit i is the
coefficient of X^i.  All hexadecimal strings are lowercase and include leading
zeroes to their stated width.

Let k=151 and let

  K = GF(2)[X]/(M),   M mask = 0x8f990819e1d470f3fe66120771ddc1056db41f.

M is monic irreducible of degree k.  Addition in K is bitwise XOR and
multiplication is carryless polynomial multiplication reduced modulo M.  For
q in K, its absolute trace is

  Tr(q) = q + q^2 + q^(2^2) + ... + q^(2^(k-1)),

which is either 0 or 1.

Let L=K[U]/(U^5+U^2+1), so every element of L has a unique coordinate vector
(q0,q1,q2,q3,q4) meaning q0+q1*U+...+q4*U^4.  Put

  d = 2^(4k)+2^(3k)+2^(2k)+2^k-1.

Define F:L->L by

  F(y) = y^3  if y lies in K (coordinates q1=q2=q3=q4=0),
  F(y) = y^d  otherwise.

This is the g(y)=y^3 case of the paper's piecewise permut

## 3. Rebuild the instance

`verify()` needs the instance object, not the question string. The CSV carries the `seed`
and `params` that produced the row, so the family's generator rebuilds it exactly.

In [ ]:
gen_path = glob.glob(os.path.join(REPO_DIR, "results", row["paper"], "gen_*.py"))[0]
spec = importlib.util.spec_from_file_location(os.path.basename(gen_path)[:-3], gen_path)
fam  = importlib.util.module_from_spec(spec)
with contextlib.redirect_stdout(io.StringIO()):   # some generators print on import
    spec.loader.exec_module(fam)

inst = fam.make_instance(seed=int(row["seed"]), **json.loads(row["params"]))

## 4. Ask the model

In [ ]:
def ask(prompt, model=MODEL):
    """One OpenRouter call. The rendered question is the whole user message, no extra framing."""
    body = json.dumps({"model": model, "reasoning": {"effort": EFFORT}, "max_tokens": MAX_TOKENS,
                       "messages": [{"role": "user", "content": prompt}]}).encode()
    req = urllib.request.Request("https://openrouter.ai/api/v1/chat/completions", data=body,
                                 headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}",
                                          "Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=1800) as resp:
        return json.load(resp)["choices"][0]["message"]["content"]

reply = ask(row["question"])
print(reply[-1200:])

## 5. Grade

`parse_answer()` returns `None` when the reply has no well-formed answer, which usually means
the model ignored the required output format. `verify(inst, answer)` returns `(bool, reason)`
and is the authority.

In [ ]:
answer = fam.parse_answer(reply)
correct, reason = fam.verify(inst, answer) if answer is not None else (False, "no well-formed answer in the reply")

print(f"{row['id']}  {MODEL}")
print(f"parsed : {answer is not None}")
print(f"correct: {correct}")
print(f"reason : {reason}")

## Notes

**This is the easy rung.** Every row here is each family's easiest preset, where the
generator-verifier gap is narrowest. A model doing well on this slice tells you little
about the hard presets.

**To evaluate at scale**, loop over rows and reuse the loaded module per family —
importing is the slow part, and some generators take seconds per instance. Group by
`paper` so each module is imported once.

**To evaluate uncontaminated**, do not use this file at all: the answers are public.
Reconstruct fresh instances straight from the generators with unseen seeds —
`fam.make_instance(seed=<new>, **fam.DIFFICULTY['hard'])` — and grade the same way.